In [ ]:
# === SETUP: instalar dependências e validar import ===
import sys, subprocess

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Tente importar; se falhar, instala e tenta de novo
try:
    from ortools.sat.python import cp_model  # noqa: F401
except Exception:
    print("Instalando pacotes: ortools e openpyxl ...")
    pip_install("ortools", "openpyxl")
    from ortools.sat.python import cp_model  # noqa: F401

import pandas as pd
print("OK! ortools e pandas disponíveis.")


OK! ortools e pandas disponíveis.


In [ ]:
# === Localizar e carregar o Excel de respostas, com fallback para upload no Colab ===
from pathlib import Path
import sys, glob
import pandas as pd

# 1) Tente caminhos conhecidos
CANDIDATES = [
    "/mnt/content/respostas_formulario.xlsx",
    "/mnt/content/respostas_formulario.xlsx",
    "/mnt/content/respostas*.xlsx",
    "/content/respostas_formulario.xlsx",
    "/content/respostas*.xlsx",
    "respostas_formulario.xlsx",
    "respostas*.xlsx",
]

found_path = None
for pat in CANDIDATES:
    for p in glob.glob(pat):
        if Path(p).is_file():
            found_path = Path(p)
            break
    if found_path:
        break

# 2) Se não encontrou, tenta abrir seletor do Colab (se disponível)
if found_path is None:
    try:
        from google.colab import files  # type: ignore
        print("Arquivo não encontrado. Abra o seletor e envie o .xlsx das respostas…")
        uploaded = files.upload()  # escolha o arquivo .xlsx
        if uploaded:
            first_name = next(iter(uploaded))
            found_path = Path(first_name)
    except Exception:
        pass  # não está no Colab ou falhou upload

# 3) Último recurso: listar o que existe nas pastas comuns para você conferir
if found_path is None:
    print("Não encontrei o arquivo automaticamente.")
    print("Arquivos em /mnt/data:")
    try:
        print("\n".join(sorted(str(p) for p in Path("/mnt/data").glob("*"))))
    except Exception:
        print("(sem acesso a /mnt/data)")

    print("\nArquivos na pasta atual:")
    print("\n".join(sorted(str(p) for p in Path(".").glob("*"))))
    raise FileNotFoundError(
        "Envie ou mova o arquivo de respostas e ajuste o caminho manualmente na variável found_path."
    )

print(f"Usando arquivo: {found_path}")

# 4) Carregar o Excel
df = pd.read_excel(found_path)
print("Prévia do dataframe carregado:")
display(df.head(121))


Usando arquivo: /content/respostas_formulario.xlsx
Prévia do dataframe carregado:


,id,email,turno,termo,curso_atual,curso_pretendido,horario,disciplinas_semestre_par,disciplinas_semestre_impar,disciplinas_interesse
0,3,guilherme.campos24@unifesp.br,integral,1,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos de Bioinformática"",""Algoritmos e ...","[""Algoritmos de Bioinformática"",""Algoritmos e ...","[""Nenhuma""]"
1,9,ruan.carvalho08@unifesp.br,integral,2,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Séries ...","[""Cálculo em várias variáveis"",""Probabilidade ...","[""Ciência de Dados II""]"
2,10,gcsilva06@unifesp.br,integral,4,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados II"",""Banco ...","[""Análise Real I"",""Álgebra Linear Computaciona...","[""Resolução de Problemas via Modelagem Matemát..."
3,11,jmc.nordemann@unifesp.br,noturno,6,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Programação Orientada a Objetos"",""Projeto e ...","[""Prática em Projetos Extensionistas I"",""Circu...",[]
4,12,maria.chain@unifesp.br,integral,2,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Fenômen...","[""Algoritmos e Estruturas de Dados II"",""Probab...","[""Ciência de Dados II"",""Comunicação Científica..."
...,...,...,...,...,...,...,...,...,...,...
115,192,karin.costa@unifesp.br,integral,2,BIOTEC,NaN,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...",[],[],[]
116,193,felipe.cezarini23@unifesp.br,integral,6,BCIT,MatComp,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Equações Diferenciais Ordinárias"",""Inferênci...","[""Análise Real I"",""Otimização Linear"",""Prática...","[""Gestão Estratégica da Tecnologia e Inovação""..."
117,194,joao.garlopa@unifesp.br,integral,6,BCIT,MatComp,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Inferência e Análise de Regressão"",""Resoluçã...","[""Trabalho de Graduação I"",""Espaços Métricos"",...","[""Resolução de Problemas via Modelagem Matemát..."
118,195,rafael.teixeira13@unifesp.br,integral,2,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Introdu...","[""Álgebra Linear"",""Cálculo em várias variáveis...","[""Otimização e Dinâmica em Economia"",""Escrita ..."



## **TESTE UM**

In [ ]:
# Modelo BCT-Noturno — Timetabling com OR-Tools (CP-SAT)
# ------------------------------------------------------
# - Lê /content/respostas_formulario.xlsx
# - Extrai slots a partir do JSON em 'horario'
# - Extrai disciplinas a partir das listas em texto
# - Constrói um modelo CP-SAT simplificado e resolve
# - Exporta /content/resultado_schedule.csv e /content/resultado_matriculas.csv

from __future__ import annotations
import json, re
from typing import List
from pathlib import Path

import pandas as pd
from ortools.sat.python import cp_model

# ============== 0) (opcional) instalar libs se faltar ==============
# !pip install -q ortools openpyxl

# ======================== 1) Carregar dados ========================
DATA_PATH = Path("/content/respostas_formulario.xlsx")  # AJUSTE SE PRECISAR
assert DATA_PATH.exists(), f"Não encontrei o arquivo: {DATA_PATH}"

df = pd.read_excel(DATA_PATH)
print("Prévia dos dados:")
print(df.head(10))
print(f"\nTotal de respostas lidas: {len(df)}")

# ================= 2) Helpers e normalizações gerais ================
DAY_ORDER = ["Segunda","Terça","Quarta","Quinta","Sexta","Sábado","Domingo"]
DAY_ABBR  = {"Segunda":"Seg","Terça":"Ter","Quarta":"Qua","Quinta":"Qui",
             "Sexta":"Sex","Sábado":"Sáb","Domingo":"Dom"}

def clean_time_label(timestr: str) -> str:
    # "08h00 - 10h00" -> "08-10"
    if not isinstance(timestr, str):
        return str(timestr)
    m = re.findall(r"(\d{2})h?\d{2}", timestr)
    if len(m) >= 2:
        return f"{m[0]}-{m[1]}"
    tmp = timestr.replace("h00","").replace("h",":").replace(" ", "")
    return tmp

def parse_horario_cell(cell):
    """cell: string JSON como:
       [{"time":"08h00 - 10h00","Segunda":true,"Terça":false,...}, ...]"""
    if isinstance(cell, list):
        return cell
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    s = str(cell).strip()
    if not s:
        return []
    try:
        return json.loads(s)
    except Exception:
        s2 = s.replace("'", '"')
        try:
            return json.loads(s2)
        except Exception:
            return []

def parse_list_cell(cell):
    """Tenta json.loads; se falhar, faz split bruto."""
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    if isinstance(cell, list):
        return [str(x).strip() for x in cell]
    s = str(cell).strip()
    if not s or s == "[]":
        return []
    try:
        arr = json.loads(s.replace("'", '"'))
        return [str(x).strip() for x in arr]
    except Exception:
        s2 = s.strip("[]")
        parts = [p.strip().strip('"').strip("'") for p in s2.split(",") if p.strip()]
        return [p for p in parts if p]

# ================= 3) Extrair TODOS os slots a partir do JSON =========
all_slots_set = set()
for _, row in df.iterrows():
    blocks = parse_horario_cell(row.get("horario"))
    for blk in blocks:
        timestr = clean_time_label(blk.get("time", ""))
        for day in DAY_ORDER:
            if day in blk and bool(blk[day]):
                slot = f"{DAY_ABBR[day]} {timestr}"  # ex.: "Seg 08-10"
                all_slots_set.add(slot)

def slot_sort_key(slot):
    try:
        day, hh = slot.split()
        start = int(hh.split("-")[0])
        full_day = [k for k,v in DAY_ABBR.items() if v == day][0]
        return (DAY_ORDER.index(full_day), start)
    except Exception:
        return (99, 999)

T = sorted(all_slots_set, key=slot_sort_key)
print(f"\nSlots detectados (|T|={len(T)}): {T[:12]}{' ...' if len(T)>12 else ''}")

# Matriz de disponibilidade A_S (todos os alunos)
S = list(df.index)  # 0..N-1
A_S = {(s,t): 0 for s in S for t in T}
for s in S:
    blocks = parse_horario_cell(df.loc[s, "horario"]) if "horario" in df.columns else []
    for blk in blocks:
        timestr = clean_time_label(blk.get("time", ""))
        for day in DAY_ORDER:
            if day in blk and bool(blk[day]):
                slot = f"{DAY_ABBR[day]} {timestr}"
                if slot in T:
                    A_S[(s, slot)] = 1

disp_por_slot = pd.Series({t: sum(A_S[(s,t)] for s in S) for t in T}).sort_values(ascending=False)
print("\nDisponibilidade por slot (top 10):")
print(disp_por_slot.head(10))

# ========== 4) Extrair TODAS as disciplinas a partir das listas ==========
DISC_COLS = ["disciplinas_semestre_par", "disciplinas_semestre_impar", "disciplinas_interesse"]

all_disc = set()
for _, row in df.iterrows():
    for col in DISC_COLS:
        if col in df.columns:
            all_disc.update(parse_list_cell(row.get(col)))

C = sorted({d for d in all_disc if d and d.lower() != "nenhuma"})
print(f"\nTotal de disciplinas detectadas (|C|={len(C)}). Exemplos: {C[:12]}{' ...' if len(C)>12 else ''}")

# Interesse D[s,c] = 1 se c aparece em qualquer lista do aluno s
D = {(s,c): 0 for s in S for c in C}
for s in S:
    interests = set()
    for col in DISC_COLS:
        if col in df.columns:
            interests.update(parse_list_cell(df.loc[s, col]))
    for c in C:
        if c in interests:
            D[(s,c)] = 1

def weight_for(cname: str) -> int:
    cn = str(cname).lower()
    if "obrig" in cn:
        return 3
    if "prior" in cn or "necess" in cn:
        return 2
    return 1

W = {(s,c): weight_for(c) for s in S for c in C}
CAP_c = {c: 40 for c in C}      # capacidade padrão (ajuste livre)
K_by_c = {c: 1 for c in C}      # 1 turma por disciplina (ajuste p/ 2, 3, ...)

print(f"\nResumo dos conjuntos: |S|={len(S)} alunos, |C|={len(C)} disciplinas, |T|={len(T)} slots")

# ====================== 5) Modelo CP-SAT (simplificado) ======================
model = cp_model.CpModel()

# z[c,k,t]: turma k de c ofertada no slot t
z = {}
for c in C:
    for k in range(K_by_c[c]):
        for t in T:
            z[(c,k,t)] = model.NewBoolVar(f"z[{c}|{k}|{t}]")

# x[s,c,k]: aluno s matriculado em (c,k)
x = {}
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            x[(s,c,k)] = model.NewBoolVar(f"x[{s}|{c}|{k}]")

# w[s,c,k,t] = x AND z (evita choque de horário por slot)
w = {}
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            for t in T:
                w[(s,c,k,t)] = model.NewBoolVar(f"w[{s}|{c}|{k}|{t}]")
                model.Add(w[(s,c,k,t)] <= x[(s,c,k)])
                model.Add(w[(s,c,k,t)] <= z[(c,k,t)])
                model.Add(w[(s,c,k,t)] >= x[(s,c,k)] + z[(c,k,t)] - 1)

# Cada turma escolhe no máx. 1 slot
for c in C:
    for k in range(K_by_c[c]):
        model.Add(sum([z[(c, k, t)] for t in T]) <= 1)

# Capacidade por turma (liga x a z)
for c in C:
    for k in range(K_by_c[c]):
        model.Add(
            sum([x[(s, c, k)] for s in S]) <=
            sum([z[(c, k, t)] for t in T]) * CAP_c[c]
        )

# Interesse e disponibilidade
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            model.Add(x[(s, c, k)] <= D[(s, c)])  # precisa ter interesse
            model.Add(  # precisa existir ao menos um slot compatível que a turma abriu
                x[(s, c, k)] <= sum([A_S[(s, t)] * z[(c, k, t)] for t in T])
            )

# Sem sobreposição por slot para o mesmo aluno
for s in S:
    for t in T:
        terms = [w[(s, c, k, t)] for c in C for k in range(K_by_c[c])]
        model.Add(sum(terms) <= 1)

# =========================== 6) Objetivo ===========================
def is_noturno(tname: str) -> bool:
    txt = str(tname)
    return ("19" in txt) or ("21" in txt) or ("19–21" in txt) or ("21–23" in txt)

lambda_noct = 0  # aumente >0 para penalizar slots fora do noturno

obj_terms = []
# Maximiza satisfação
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            obj_terms.append(W[(s,c)] * x[(s,c,k)])

# Penaliza slots fora do noturno (opcional)
if lambda_noct > 0:
    for c in C:
        for k in range(K_by_c[c]):
            for t in T:
                if not is_noturno(t):
                    obj_terms.append(-lambda_noct * z[(c,k,t)])

model.Maximize(sum(obj_terms))

# =========================== 7) Resolver ===========================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8
status = solver.Solve(model)
print("\nStatus:", solver.StatusName(status))

# ======================== 8) Exportar CSVs =========================
OUT_DIR = Path("/content")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Oferta (schedule)
rows_sched = []
for c in C:
    for k in range(K_by_c[c]):
        chosen_slots = [t for t in T if solver.Value(z[(c,k,t)]) == 1]
        rows_sched.append({
            "disciplina": c,
            "turma": k,
            "slots_escolhidos": ", ".join(map(str, chosen_slots)) if chosen_slots else ""
        })
df_sched = pd.DataFrame(rows_sched)
df_sched.to_csv(OUT_DIR / "resultado_schedule.csv", index=False)

# Matrículas
rows_mat = []
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            if solver.Value(x[(s,c,k)]) == 1:
                rows_mat.append({"aluno_idx": s, "disciplina": c, "turma": k})
df_mat = pd.DataFrame(rows_mat)
df_mat.to_csv(OUT_DIR / "resultado_matriculas.csv", index=False)

print("\nArquivos gerados:")
print(" -", OUT_DIR / "resultado_schedule.csv")
print(" -", OUT_DIR / "resultado_matriculas.csv")

# Visualização rápida (se estiver em notebook)
try:
    from IPython.display import display
    print("\nPrévia do schedule:")
    display(df_sched.head(10))
    print("\nPrévia das matrículas:")
    display(df_mat.head(10))
except Exception:
    pass

# Quantas turmas abriram
turmas_abertas = sum(1 for c in C for k in range(K_by_c[c]) if any(solver.Value(z[(c,k,t)]) for t in T))

# Total de matrículas
total_matriculas = sum(1 for s in S for c in C for k in range(K_by_c[c]) if solver.Value(x[(s,c,k)])==1)

# % de interesses atendidos (entre todos D[s,c]==1)
total_interesses = sum(D[(s,c)] for s in S for c in C)
atendidos = sum(1 for s in S for c in C for k in range(K_by_c[c]) if D[(s,c)]==1 and solver.Value(x[(s,c,k)])==1)
pct = (atendidos/total_interesses*100) if total_interesses else 0.0

print("Turmas abertas:", turmas_abertas)
print("Total de matrículas:", total_matriculas)
print(f"% de interesses atendidos: {pct:.1f}%")

# ===== KPI extra: % de NOTURNO (oferta e matrículas) =====

# 1) Mapa (c,k) -> slot escolhido
slot_of = {}
for c in C:
    for k in range(K_by_c[c]):
        chosen = [t for t in T if solver.Value(z[(c,k,t)]) == 1]
        slot_of[(c,k)] = chosen[0] if chosen else None

# 2) % de turmas abertas no noturno
turmas_abertas = sum(1 for ck, t in slot_of.items() if t is not None)
turmas_noturno = sum(1 for ck, t in slot_of.items() if (t is not None and is_noturno(t)))
pct_turmas_noturno = (100 * turmas_noturno / turmas_abertas) if turmas_abertas else 0.0

# 3) % de matrículas no noturno
total_mats = 0
mats_noturno = 0
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            if solver.Value(x[(s,c,k)]) == 1:
                total_mats += 1
                tck = slot_of[(c,k)]
                if tck is not None and is_noturno(tck):
                    mats_noturno += 1
pct_mats_noturno = (100 * mats_noturno / total_mats) if total_mats else 0.0

print("\n--- KPI Noturno ---")
print(f"Turmas abertas no noturno: {turmas_noturno}/{turmas_abertas}  ({pct_turmas_noturno:.1f}%)")
print(f"Matrículas no noturno:     {mats_noturno}/{total_mats}      ({pct_mats_noturno:.1f}%)")

# (Opcional) anotar o período nos CSVs já gerados:
if 'df_sched' in globals():
    df_sched['periodo'] = df_sched['slots_escolhidos'].apply(
        lambda s: "Noturno" if any(is_noturno(t.strip()) for t in str(s).split(",") if t) else "Diurno/Outro"
    )
    df_sched.to_csv(OUT_DIR / "resultado_schedule.csv", index=False)

if 'df_mat' in globals():
    # adiciona a coluna 'slot' para cada matrícula, e o rótulo Noturno/Diurno
    def slot_for_row(row):
        return slot_of.get((row['disciplina'], row['turma']), None)
    # se df_mat tem colunas 'disciplina' e 'turma' como strings/ints, ajusta o acesso:
    df_mat['slot'] = [slot_of[(row['disciplina'], row['turma'])] for _, row in df_mat.iterrows()]
    df_mat['periodo'] = df_mat['slot'].apply(lambda t: "Noturno" if (t is not None and is_noturno(t)) else "Diurno/Outro")
    df_mat.to_csv(OUT_DIR / "resultado_matriculas.csv", index=False)




Prévia dos dados:
   id                          email     turno  termo curso_atual  \
0   3  guilherme.campos24@unifesp.br  integral      1        BCIT   
1   9     ruan.carvalho08@unifesp.br  integral      2        BCIT   
2  10           gcsilva06@unifesp.br  integral      4        BCIT   
3  11       jmc.nordemann@unifesp.br   noturno      6        BCIT   
4  12         maria.chain@unifesp.br  integral      2        BCIT   
5  16     nichollas.lohan@unifesp.br  integral      7        BCIT   
6  24          evsribeiro@unifesp.br   noturno      2        BCIT   
7  25        felipe.bleck@unifesp.br  integral      4        BCIT   
8  26     vitor.peneluppi@unifesp.br  integral      6        BCIT   
9  35        joao.zampoli@unifesp.br  integral      6        BCIT   

  curso_pretendido                                            horario  \
0            CCOMP  [{"time":"08h00 - 10h00","Segunda":true,"Terça...   
1            CCOMP  [{"time":"08h00 - 10h00","Segunda":true,"Terça...   
2  

,disciplina,turma,slots_escolhidos
0,Algoritmos de Bioinformática,0,
1,Algoritmos e Estruturas de Dados I,0,
2,Algoritmos e Estruturas de Dados II,0,
3,Algoritmos em Bioinformatica,0,
4,Anatomia,0,
5,Análise Real I,0,
6,Análise Real II,0,
7,Análise de Investimento e Risco,0,
8,Análise de Sinais,0,
9,Aplicações de Sistemas Dinâmicos a Ciência e T...,0,



Prévia das matrículas:


""


Turmas abertas: 198
Total de matrículas: 0
% de interesses atendidos: 0.0%

--- KPI Noturno ---
Turmas abertas no noturno: 0/0  (0.0%)
Matrículas no noturno:     0/0      (0.0%)


## **TESTE DOIS**

In [ ]:
# ==========================================================
# BCT-Noturno — Timetabling com OR-Tools (CP-SAT) — Versão Unificada
# ==========================================================
# - Setup/instalação, carga do Excel com fallback
# - Parsing de 'horario' (JSON) e das listas de disciplinas
# - K_by_c por DEMANDA (ceil(demanda/cap)), limite máx 3 turmas
# - Proíbe 2 turmas da mesma disciplina no mesmo slot
# - Prioriza alunos do noturno e penaliza (opcional) fora do noturno
# - Limita nº de disciplinas por aluno (LOAD_MAX)
# - Bônus por slots com alta disponibilidade
# - Solver com +tempo e KPIs
# - Exporta /content/resultado_schedule.csv e /content/resultado_matriculas.csv
# ==========================================================

# ---------- 0) SETUP: instalar dependências se faltar ----------
import sys, subprocess
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    from ortools.sat.python import cp_model  # noqa
except Exception:
    print("Instalando pacotes: ortools e openpyxl ...")
    pip_install("ortools", "openpyxl")
    from ortools.sat.python import cp_model  # noqa

import pandas as pd
import json, re, glob
from pathlib import Path
from typing import List

# ---------- 1) Localizar e carregar o Excel (com fallback Colab) ----------
CANDIDATES = [
    "/content/respostas_formulario.xlsx",
    "/content/respostas_formulario1.xlsx",
    "/content/*respostas*formulario*.xlsx",
    "respostas_formulario.xlsx",
    "respostas_formulario1.xlsx",
    "*respostas*formulario*.xlsx",
]

found_path = None
for pat in CANDIDATES:
    hits = [Path(p) for p in glob.glob(pat)]
    if hits:
        found_path = hits[0]
        break

if found_path is None:
    try:
        from google.colab import files  # type: ignore
        print("Arquivo não encontrado. Selecione o .xlsx…")
        uploaded = files.upload()
        if uploaded:
            first_name = next(iter(uploaded))
            found_path = Path(first_name)
    except Exception:
        pass

if found_path is None:
    raise FileNotFoundError("Não achei o .xlsx. Faça upload pela aba Files ou use files.upload().")

print(f"Usando arquivo: {found_path}")
df = pd.read_excel(found_path)
print("Prévia dos dados:")
display(df.head(10))
print(f"Total de respostas lidas: {len(df)}")

# ---------- 2) Helpers gerais ----------
DAY_ORDER = ["Segunda","Terça","Quarta","Quinta","Sexta","Sábado","Domingo"]
DAY_ABBR  = {"Segunda":"Seg","Terça":"Ter","Quarta":"Qua","Quinta":"Qui",
             "Sexta":"Sex","Sábado":"Sáb","Domingo":"Dom"}

def clean_time_label(timestr: str) -> str:
    # "08h00 - 10h00" -> "08-10"
    if not isinstance(timestr, str):
        return str(timestr)
    m = re.findall(r"(\d{2})h?\d{2}", timestr)
    if len(m) >= 2:
        return f"{m[0]}-{m[1]}"
    tmp = timestr.replace("h00","").replace("h",":").replace(" ", "")
    return tmp

def parse_horario_cell(cell):
    """Ex.: [{"time":"08h00 - 10h00","Segunda":true,...}, ...]"""
    if isinstance(cell, list):
        return cell
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    s = str(cell).strip()
    if not s:
        return []
    try:
        return json.loads(s)
    except Exception:
        s2 = s.replace("'", '"')
        try:
            return json.loads(s2)
        except Exception:
            return []

def parse_list_cell(cell):
    """Abre strings tipo '["Disc A","Disc B"]' ou faz split robusto."""
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    if isinstance(cell, list):
        return [str(x).strip() for x in cell]
    s = str(cell).strip()
    if not s or s == "[]":
        return []
    try:
        arr = json.loads(s.replace("'", '"'))
        return [str(x).strip() for x in arr]
    except Exception:
        s2 = s.strip("[]")
        parts = [p.strip().strip('"').strip("'") for p in s2.split(",") if p.strip()]
        return [p for p in parts if p]

# ---------- 3) Slots T e matriz de disponibilidade A_S ----------
all_slots_set = set()
if "horario" not in df.columns:
    raise ValueError("Coluna 'horario' não encontrada no Excel.")

for _, row in df.iterrows():
    blocks = parse_horario_cell(row["horario"])
    for blk in blocks:
        timestr = clean_time_label(blk.get("time", ""))
        for day in DAY_ORDER:
            if day in blk and bool(blk[day]):
                slot = f"{DAY_ABBR[day]} {timestr}"
                all_slots_set.add(slot)

def slot_sort_key(slot):
    try:
        day, hh = slot.split()
        start = int(hh.split("-")[0])
        full_day = [k for k,v in DAY_ABBR.items() if v == day][0]
        return (DAY_ORDER.index(full_day), start)
    except Exception:
        return (99, 999)

T = sorted(all_slots_set, key=slot_sort_key)
print(f"\nSlots detectados (|T|={len(T)}): {T[:12]}{' ...' if len(T)>12 else ''}")

S = list(df.index)  # 0..N-1
A_S = {(s,t): 0 for s in S for t in T}
for s in S:
    blocks = parse_horario_cell(df.loc[s, "horario"])
    for blk in blocks:
        timestr = clean_time_label(blk.get("time", ""))
        for day in DAY_ORDER:
            if day in blk and bool(blk[day]):
                slot = f"{DAY_ABBR[day]} {timestr}"
                if slot in T:
                    A_S[(s, slot)] = 1

disp_por_slot = pd.Series({t: sum(A_S[(s,t)] for s in S) for t in T}).sort_values(ascending=False)
print("\nDisponibilidade por slot (top 10):")
display(disp_por_slot.head(10))

# ---------- 4) Disciplinas C e interesse D ----------
DISC_COLS = ["disciplinas_semestre_par", "disciplinas_semestre_impar", "disciplinas_interesse"]

all_disc = set()
for _, row in df.iterrows():
    for col in DISC_COLS:
        if col in df.columns:
            all_disc.update(parse_list_cell(row.get(col)))

C = sorted({d for d in all_disc if d and d.lower() != "nenhuma"})
print(f"\nTotal de disciplinas detectadas (|C|={len(C)}). Exemplos: {C[:12]}{' ...' if len(C)>12 else ''}")

D = {(s,c): 0 for s in S for c in C}
for s in S:
    interests = set()
    for col in DISC_COLS:
        if col in df.columns:
            interests.update(parse_list_cell(df.loc[s, col]))
    for c in C:
        if c in interests:
            D[(s,c)] = 1

# Pesos W (com bônus para aluno noturno)
def weight_for_disc(cname: str) -> int:
    cn = str(cname).lower()
    if "obrig" in cn:
        return 3
    if "prior" in cn or "necess" in cn:
        return 2
    return 1

aluno_noturno = {s: (str(df.loc[s, "turno"]).strip().lower()=="noturno" if "turno" in df.columns else False) for s in S}
def weight_for_personalizado(s, c):
    base = weight_for_disc(c)
    return base + 1 if aluno_noturno[s] else base  # prioriza noturno

W = {(s,c): weight_for_personalizado(s,c) for s in S for c in C}

# Capacidades por disciplina (padrão)
CAP_c = {c: 40 for c in C}
# Ex.: aumentar capacidade em gargalos (exemplos, ajuste como quiser)
for c in C:
    if ("Cálculo" in c) or ("Algoritmos e Estruturas de Dados I" in c):
        CAP_c[c] = 60

# ---------- 5) Nº de turmas por demanda (limite máx 3) ----------
demanda = {c: sum(D[(s,c)] for s in S) for c in C}
K_by_c = {}
for c in C:
    cap = CAP_c.get(c, 40)
    # abre ceil(demanda/cap), no mínimo 1:
    K_by_c[c] = max(1, (demanda[c] + cap - 1) // cap)
    # limite superior:
    K_by_c[c] = min(K_by_c[c], 3)

print("\nExemplos de (demanda, cap, turmas) nas 10 maiores demandas:")
for c_top, _ in sorted(demanda.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"- {c_top}: demanda={demanda[c_top]}, cap={CAP_c[c_top]}, turmas={K_by_c[c_top]}")

# ---------- 6) Modelo CP-SAT ----------
model = cp_model.CpModel()

# z[c,k,t]: turma k de c no slot t
z = {}
for c in C:
    for k in range(K_by_c[c]):
        for t in T:
            z[(c,k,t)] = model.NewBoolVar(f"z[{c}|{k}|{t}]")

# x[s,c,k]: aluno s matriculado em (c,k)
x = {}
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            x[(s,c,k)] = model.NewBoolVar(f"x[{s}|{c}|{k}]")

# w[s,c,k,t] = x AND z
w = {}
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            for t in T:
                w[(s,c,k,t)] = model.NewBoolVar(f"w[{s}|{c}|{k}|{t}]")
                model.Add(w[(s,c,k,t)] <= x[(s,c,k)])
                model.Add(w[(s,c,k,t)] <= z[(c,k,t)])
                model.Add(w[(s,c,k,t)] >= x[(s,c,k)] + z[(c,k,t)] - 1)

# Cada turma escolhe no máx. 1 slot
for c in C:
    for k in range(K_by_c[c]):
        model.Add(sum(z[(c, k, t)] for t in T) <= 1)

# Proibir 2 turmas da mesma disciplina no mesmo slot
for c in C:
    for t in T:
        model.Add(sum(z[(c, k, t)] for k in range(K_by_c[c])) <= 1)

# Capacidade por turma
for c in C:
    for k in range(K_by_c[c]):
        model.Add(
            sum(x[(s, c, k)] for s in S) <=
            sum(z[(c, k, t)] for t in T) * CAP_c[c]
        )

# Interesse e disponibilidade
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            model.Add(x[(s, c, k)] <= D[(s, c)])  # precisa de interesse
            model.Add(x[(s, c, k)] <= sum(A_S[(s, t)] * z[(c, k, t)] for t in T))  # precisa de slot compatível

# Sem sobreposição por slot para o mesmo aluno
for s in S:
    for t in T:
        model.Add(sum(w[(s, c, k, t)] for c in C for k in range(K_by_c[c])) <= 1)

# Limitar nº de disciplinas por aluno (espalha melhor as vagas)
LOAD_MIN, LOAD_MAX = 0, 6  # ajuste conforme política
for s in S:
    model.Add(sum(x[(s,c,k)] for c in C for k in range(K_by_c[c])) <= LOAD_MAX)
    # model.Add(sum(x[(s,c,k)] for c in C for k in range(K_by_c[c])) >= LOAD_MIN)

# ---------- 7) Objetivo: satisfação + preferências de slot ----------
def is_noturno(tname: str) -> bool:
    txt = str(tname)
    # Ajuste sua definição de noturno; aqui 19-23
    return ("19" in txt) or ("21" in txt)

lambda_noct = 1.0  # penaliza fora do noturno (aumente/diminua)
obj_terms = []

# satisfação (W * x)
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            obj_terms.append(W[(s,c)] * x[(s,c,k)])

# penalidade por slots fora do noturno
for c in C:
    for k in range(K_by_c[c]):
        for t in T:
            if not is_noturno(t):
                obj_terms.append(-lambda_noct * z[(c,k,t)])

# bônus por slots com alta disponibilidade (suave, só para "empurrar")
if T:
    max_disp = max(sum(A_S[(s,t)] for s in S) for t in T)
    bonus_slot = {t: (sum(A_S[(s,t)] for s in S) / max_disp) for t in T}
    slot_weight = 0.05
    for c in C:
        for k in range(K_by_c[c]):
            for t in T:
                obj_terms.append(slot_weight * bonus_slot[t] * z[(c,k,t)])

model.Maximize(sum(obj_terms))

# ---------- 8) Resolver ----------
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 180.0   # dá mais tempo para qualidade
solver.parameters.num_search_workers = 8
status = solver.Solve(model)
print("\nStatus:", solver.StatusName(status))

# ---------- 9) Exportar resultados ----------
OUT_DIR = Path("/content")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Oferta (schedule)
rows_sched = []
for c in C:
    for k in range(K_by_c[c]):
        chosen_slots = [t for t in T if solver.Value(z[(c,k,t)]) == 1]
        rows_sched.append({
            "disciplina": c,
            "turma": k,
            "slots_escolhidos": ", ".join(map(str, chosen_slots)) if chosen_slots else ""
        })
df_sched = pd.DataFrame(rows_sched)
df_sched.to_csv(OUT_DIR / "resultado_schedule.csv", index=False)

# Matrículas
rows_mat = []
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            if solver.Value(x[(s,c,k)]) == 1:
                rows_mat.append({"aluno_idx": s, "disciplina": c, "turma": k})
df_mat = pd.DataFrame(rows_mat)
df_mat.to_csv(OUT_DIR / "resultado_matriculas.csv", index=False)

print("\nArquivos gerados:")
print(" -", OUT_DIR / "resultado_schedule.csv")
print(" -", OUT_DIR / "resultado_matriculas.csv")

try:
    from IPython.display import display
    print("\nPrévia do schedule:")
    display(df_sched.head(12))
    print("\nPrévia das matrículas:")
    display(df_mat.head(12))
except Exception:
    pass

# ---------- 10) KPIs úteis ----------
total_interesses = sum(D[(s,c)] for s in S for c in C)
atendidos = 0
for s in S:
    for c in C:
        if D[(s,c)]==1:
            if any(solver.Value(x[(s,c,k)])==1 for k in range(K_by_c[c])):
                atendidos += 1

pct = (atendidos/total_interesses*100) if total_interesses else 0.0
turmas_abertas = sum(1 for c in C for k in range(K_by_c[c]) if any(solver.Value(z[(c,k,t)]) for t in T))

print("\n--- KPIs ---")
print("Turmas abertas:", turmas_abertas)
print("Total de matrículas:", len(df_mat))
print(f"% de interesses atendidos: {pct:.1f}%")

# ===== KPI extra: % de NOTURNO (oferta e matrículas) =====

# 1) Mapa (c,k) -> slot escolhido
slot_of = {}
for c in C:
    for k in range(K_by_c[c]):
        chosen = [t for t in T if solver.Value(z[(c,k,t)]) == 1]
        slot_of[(c,k)] = chosen[0] if chosen else None

# 2) % de turmas abertas no noturno
turmas_abertas = sum(1 for ck, t in slot_of.items() if t is not None)
turmas_noturno = sum(1 for ck, t in slot_of.items() if (t is not None and is_noturno(t)))
pct_turmas_noturno = (100 * turmas_noturno / turmas_abertas) if turmas_abertas else 0.0

# 3) % de matrículas no noturno
total_mats = 0
mats_noturno = 0
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            if solver.Value(x[(s,c,k)]) == 1:
                total_mats += 1
                tck = slot_of[(c,k)]
                if tck is not None and is_noturno(tck):
                    mats_noturno += 1
pct_mats_noturno = (100 * mats_noturno / total_mats) if total_mats else 0.0

print("\n--- KPI Noturno ---")
print(f"Turmas abertas no noturno: {turmas_noturno}/{turmas_abertas}  ({pct_turmas_noturno:.1f}%)")
print(f"Matrículas no noturno:     {mats_noturno}/{total_mats}      ({pct_mats_noturno:.1f}%)")

# (Opcional) anotar o período nos CSVs já gerados:
if 'df_sched' in globals():
    df_sched['periodo'] = df_sched['slots_escolhidos'].apply(
        lambda s: "Noturno" if any(is_noturno(t.strip()) for t in str(s).split(",") if t) else "Diurno/Outro"
    )
    df_sched.to_csv(OUT_DIR / "resultado_schedule.csv", index=False)

if 'df_mat' in globals():
    # adiciona a coluna 'slot' para cada matrícula, e o rótulo Noturno/Diurno
    def slot_for_row(row):
        return slot_of.get((row['disciplina'], row['turma']), None)
    # se df_mat tem colunas 'disciplina' e 'turma' como strings/ints, ajusta o acesso:
    df_mat['slot'] = [slot_of[(row['disciplina'], row['turma'])] for _, row in df_mat.iterrows()]
    df_mat['periodo'] = df_mat['slot'].apply(lambda t: "Noturno" if (t is not None and is_noturno(t)) else "Diurno/Outro")
    df_mat.to_csv(OUT_DIR / "resultado_matriculas.csv", index=False)



Usando arquivo: /content/respostas_formulario.xlsx
Prévia dos dados:


,id,email,turno,termo,curso_atual,curso_pretendido,horario,disciplinas_semestre_par,disciplinas_semestre_impar,disciplinas_interesse
0,3,guilherme.campos24@unifesp.br,integral,1,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos de Bioinformática"",""Algoritmos e ...","[""Algoritmos de Bioinformática"",""Algoritmos e ...","[""Nenhuma""]"
1,9,ruan.carvalho08@unifesp.br,integral,2,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Séries ...","[""Cálculo em várias variáveis"",""Probabilidade ...","[""Ciência de Dados II""]"
2,10,gcsilva06@unifesp.br,integral,4,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados II"",""Banco ...","[""Análise Real I"",""Álgebra Linear Computaciona...","[""Resolução de Problemas via Modelagem Matemát..."
3,11,jmc.nordemann@unifesp.br,noturno,6,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Programação Orientada a Objetos"",""Projeto e ...","[""Prática em Projetos Extensionistas I"",""Circu...",[]
4,12,maria.chain@unifesp.br,integral,2,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Fenômen...","[""Algoritmos e Estruturas de Dados II"",""Probab...","[""Ciência de Dados II"",""Comunicação Científica..."
5,16,nichollas.lohan@unifesp.br,integral,7,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Algoritmos e Estruturas de Dados I"",""Análise...","[""Algoritmos e Estruturas de Dados I"",""Álgebra...","[""Simulação Computacional Aplicada em Saúde"",""..."
6,24,evsribeiro@unifesp.br,noturno,2,BCIT,Só BCT,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Anatomia"",""Ciência, Tecnologia, Sociedade e ...",[],[]
7,25,felipe.bleck@unifesp.br,integral,4,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Análise...","[""Algoritmos e Estruturas de Dados I"",""Circuit...","[""Otimização e Dinâmica em Economia""]"
8,26,vitor.peneluppi@unifesp.br,integral,6,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Engenharia de Software"",""Compiladores"",""Proj...",[],[]
9,35,joao.zampoli@unifesp.br,integral,6,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Programação Concorrente e Distribuída"",""Rede...","[""Projeto e Análise de Algoritmos"",""Cálculo em...",[]


Total de respostas lidas: 120

Slots detectados (|T|=36): ['Seg 08-10', 'Seg 10-12', 'Seg 13-15', 'Seg 15-17', 'Seg 19-21', 'Seg 21-23', 'Ter 08-10', 'Ter 10-12', 'Ter 13-15', 'Ter 15-17', 'Ter 19-21', 'Ter 21-23'] ...

Disponibilidade por slot (top 10):


,0
Seg 15-17,86
Qua 15-17,86
Qui 15-17,84
Ter 15-17,82
Qui 19-21,79
Ter 19-21,78
Qua 19-21,77
Seg 19-21,77
Sex 15-17,75
Qui 21-23,72



Total de disciplinas detectadas (|C|=198). Exemplos: ['Algoritmos de Bioinformática', 'Algoritmos e Estruturas de Dados I', 'Algoritmos e Estruturas de Dados II', 'Algoritmos em Bioinformatica', 'Anatomia', 'Análise Real I', 'Análise Real II', 'Análise de Investimento e Risco', 'Análise de Sinais', 'Aplicações de Sistemas Dinâmicos a Ciência e Tecnologia', 'Aprendizado de Máquina e Reconhecimento de Padrões', 'Arquitetura e organização de Computadores'] ...

Exemplos de (demanda, cap, turmas) nas 10 maiores demandas:
- Cálculo em várias variáveis: demanda=37, cap=60, turmas=1
- Séries e Equações Diferenciais Ordinárias: demanda=37, cap=40, turmas=1
- Algoritmos e Estruturas de Dados II: demanda=31, cap=60, turmas=1
- Algoritmos e Estruturas de Dados I: demanda=29, cap=60, turmas=1
- Geometria Analítica: demanda=28, cap=40, turmas=1
- Cálculo Numérico: demanda=26, cap=60, turmas=1
- Circuitos Digitais: demanda=25, cap=40, turmas=1
- Ciência de Dados II: demanda=25, cap=40, turmas=1
- C

,disciplina,turma,slots_escolhidos
0,Algoritmos de Bioinformática,0,Qui 19-21
1,Algoritmos e Estruturas de Dados I,0,Ter 08-10
2,Algoritmos e Estruturas de Dados II,0,Seg 08-10
3,Algoritmos em Bioinformatica,0,Qui 19-21
4,Anatomia,0,Qui 21-23
5,Análise Real I,0,Qui 13-15
6,Análise Real II,0,Qui 19-21
7,Análise de Investimento e Risco,0,Qua 19-21
8,Análise de Sinais,0,Sáb 21-23
9,Aplicações de Sistemas Dinâmicos a Ciência e T...,0,Qui 19-21



Prévia das matrículas:


,aluno_idx,disciplina,turma
0,0,Algoritmos e Estruturas de Dados II,0
1,1,Algoritmos e Estruturas de Dados II,0
2,1,Ciência de Dados II,0
3,1,Fenômenos Mecânicos,0
4,1,Geometria Analítica,0
5,1,Matemática Discreta,0
6,1,Álgebra Linear,0
7,2,Algoritmos e Estruturas de Dados II,0
8,2,Análise Real I,0
9,2,Banco de Dados,0



--- KPIs ---
Turmas abertas: 198
Total de matrículas: 592
% de interesses atendidos: 44.7%

--- KPI Noturno ---
Turmas abertas no noturno: 157/198  (79.3%)
Matrículas no noturno:     319/592      (53.9%)


## **TESTE TRES**

In [ ]:
# ==========================================================
# BCT-Noturno — Timetabling com OR-Tools (CP-SAT) — v2 Max Coverage
# ==========================================================
# Foco: aumentar % de interesses atendidos
# - Uma turma por aluno por disciplina
# - +Turmas por demanda (espalha conflitos), até 6
# - Limite de carga por aluno (espalha vagas)
# - Descongestionar slots + separar pares com alta co-demanda
# - Tempo de solver ampliado
# ==========================================================

# ---------- 0) SETUP ----------
import sys, subprocess
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
try:
    from ortools.sat.python import cp_model  # noqa
except Exception:
    print("Instalando pacotes: ortools e openpyxl ...")
    pip_install("ortools", "openpyxl")
    from ortools.sat.python import cp_model  # noqa

import pandas as pd
import json, re, glob
from pathlib import Path
from typing import List
from collections import defaultdict

# ---------- 1) Localizar e carregar o Excel ----------
CANDIDATES = [
    "/content/respostas_formulario.xlsx",
    "/content/respostas_formulario1.xlsx",
    "/content/*respostas*formulario*.xlsx",
    "respostas_formulario.xlsx",
    "respostas_formulario1.xlsx",
    "*respostas*formulario*.xlsx",
]
found_path = None
for pat in CANDIDATES:
    hits = [Path(p) for p in glob.glob(pat)]
    if hits:
        found_path = hits[0]; break
if found_path is None:
    try:
        from google.colab import files  # type: ignore
        print("Arquivo não encontrado. Selecione o .xlsx…")
        uploaded = files.upload()
        if uploaded:
            first_name = next(iter(uploaded))
            found_path = Path(first_name)
    except Exception:
        pass
if found_path is None:
    raise FileNotFoundError("Não achei o .xlsx. Faça upload pela aba Files ou use files.upload().")

print(f"Usando arquivo: {found_path}")
df = pd.read_excel(found_path)
print("Prévia dos dados:")
display(df.head(10))
print(f"Total de respostas lidas: {len(df)}")

# ---------- 2) Helpers ----------
DAY_ORDER = ["Segunda","Terça","Quarta","Quinta","Sexta","Sábado","Domingo"]
DAY_ABBR  = {"Segunda":"Seg","Terça":"Ter","Quarta":"Qua","Quinta":"Qui",
             "Sexta":"Sex","Sábado":"Sáb","Domingo":"Dom"}

def clean_time_label(timestr: str) -> str:
    if not isinstance(timestr, str): return str(timestr)
    m = re.findall(r"(\d{2})h?\d{2}", timestr)
    if len(m) >= 2: return f"{m[0]}-{m[1]}"
    return timestr.replace("h00","").replace("h",":").replace(" ", "")

def parse_horario_cell(cell):
    if isinstance(cell, list): return cell
    if cell is None or (isinstance(cell, float) and pd.isna(cell)): return []
    s = str(cell).strip()
    if not s: return []
    try:
        return json.loads(s)
    except Exception:
        s2 = s.replace("'", '"')
        try: return json.loads(s2)
        except Exception: return []

def parse_list_cell(cell):
    if cell is None or (isinstance(cell, float) and pd.isna(cell)): return []
    if isinstance(cell, list): return [str(x).strip() for x in cell]
    s = str(cell).strip()
    if not s or s == "[]": return []
    try:
        arr = json.loads(s.replace("'", '"'))
        return [str(x).strip() for x in arr]
    except Exception:
        s2 = s.strip("[]")
        parts = [p.strip().strip('"').strip("'") for p in s2.split(",") if p.strip()]
        return [p for p in parts if p]

# ---------- 3) Slots T e disponibilidade A_S ----------
all_slots_set = set()
if "horario" not in df.columns:
    raise ValueError("Coluna 'horario' não encontrada no Excel.")

for _, row in df.iterrows():
    blocks = parse_horario_cell(row["horario"])
    for blk in blocks:
        timestr = clean_time_label(blk.get("time", ""))
        for day in DAY_ORDER:
            if day in blk and bool(blk[day]):
                all_slots_set.add(f"{DAY_ABBR[day]} {timestr}")

def slot_sort_key(slot):
    try:
        day, hh = slot.split()
        start = int(hh.split("-")[0])
        full_day = [k for k,v in DAY_ABBR.items() if v==day][0]
        return (DAY_ORDER.index(full_day), start)
    except Exception:
        return (99, 999)

T = sorted(all_slots_set, key=slot_sort_key)
print(f"\nSlots detectados (|T|={len(T)}): {T[:12]}{' ...' if len(T)>12 else ''}")

S = list(df.index)
A_S = {(s,t): 0 for s in S for t in T}
for s in S:
    blocks = parse_horario_cell(df.loc[s,"horario"])
    for blk in blocks:
        timestr = clean_time_label(blk.get("time",""))
        for day in DAY_ORDER:
            if day in blk and bool(blk[day]):
                slot = f"{DAY_ABBR[day]} {timestr}"
                if slot in T: A_S[(s,slot)] = 1

disp_por_slot = pd.Series({t: sum(A_S[(s,t)] for s in S) for t in T}).sort_values(ascending=False)
print("\nDisponibilidade por slot (top 10):")
display(disp_por_slot.head(10))

# ---------- 4) Disciplinas C e interesse D ----------
DISC_COLS = ["disciplinas_semestre_par","disciplinas_semestre_impar","disciplinas_interesse"]
all_disc = set()
for _, row in df.iterrows():
    for col in DISC_COLS:
        if col in df.columns: all_disc.update(parse_list_cell(row.get(col)))
C = sorted({d for d in all_disc if d and d.lower()!="nenhuma"})
print(f"\nTotal de disciplinas detectadas (|C|={len(C)}). Exemplos: {C[:12]}{' ...' if len(C)>12 else ''}")

D = {(s,c): 0 for s in S for c in C}
for s in S:
    interests = set()
    for col in DISC_COLS:
        if col in df.columns: interests.update(parse_list_cell(df.loc[s,col]))
    for c in C:
        if c in interests: D[(s,c)] = 1

# Pesos (prioriza noturno levemente)
def weight_for_disc(cname: str) -> int:
    cn = str(cname).lower()
    if "obrig" in cn: return 3
    if "prior" in cn or "necess" in cn: return 2
    return 1
aluno_noturno = {s: (str(df.loc[s,"turno"]).strip().lower()=="noturno" if "turno" in df.columns else False) for s in S}
def weight_for_personalizado(s,c):
    base = weight_for_disc(c)
    return base + 1 if aluno_noturno[s] else base
W = {(s,c): weight_for_personalizado(s,c) for s in S for c in C}

# Capacidades (ex.: gargalos um pouco maiores)
CAP_c = {c: 50 for c in C}
for c in C:
    if ("Cálculo" in c) or ("Algoritmos e Estruturas de Dados" in c):
        CAP_c[c] = 60

# ---------- 5) +Turmas por demanda (agressivo p/ cobertura) ----------
demanda = {c: sum(D[(s,c)] for s in S) for c in C}
K_by_c = {}
for c in C:
    cap = CAP_c.get(c, 50)
    cap_eff = max(20, int(0.7*cap))        # "capacidade efetiva" p/ espalhar conflitos
    turmas = max(1, (demanda[c] + cap_eff - 1) // cap_eff)  # ceil(dem/cap_eff)
    if demanda[c] >= 0.6*cap:               # já “próximo” de encher
        turmas = max(turmas, 2)
    K_by_c[c] = min(turmas, 6)              # limite sup. p/ não explodir
print("\nTop 10 demandas e turmas:")
for c_top,_ in sorted(demanda.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"- {c_top}: demanda={demanda[c_top]}, cap={CAP_c[c_top]}, turmas={K_by_c[c_top]}")

# ---------- 5A) SLOT_MAX e pares fortes ----------
# SLOT_MAX mais folgado: ~ 2.5x a média de turmas por slot
media_turmas = sum(K_by_c.values()) / max(1,len(T))
SLOT_MAX = max(18, int(2.5 * media_turmas))  # ex.: 18, 20, ...
print(f"\nSLOT_MAX global (limite de turmas simultâneas por slot): {SLOT_MAX}")

# Pares com alta co-demanda: separar no mesmo slot
co_demanda = defaultdict(int)
for s in S:
    ints = [c for c in C if D[(s,c)]==1]
    for i in range(len(ints)):
        ci = ints[i]
        for j in range(i+1, len(ints)):
            cj = ints[j]
            key = (ci,cj) if ci<cj else (cj,ci)
            co_demanda[key] += 1
PAIRS_THRESHOLD = 10
MAX_PAIRS = 300
pairs_fortes = [p for p,v in co_demanda.items() if v >= PAIRS_THRESHOLD]
pairs_fortes = sorted(pairs_fortes, key=lambda p: co_demanda[p], reverse=True)[:MAX_PAIRS]
print(f"Pares fortes (co-demanda >= {PAIRS_THRESHOLD}): {len(pairs_fortes)}")

# ---------- 6) Modelo CP-SAT ----------
model = cp_model.CpModel()

# z[c,k,t]: turma k de c no slot t
z = {(c,k,t): model.NewBoolVar(f"z[{c}|{k}|{t}]")
     for c in C for k in range(K_by_c[c]) for t in T}

# x[s,c,k]: aluno s matriculado em (c,k)
x = {(s,c,k): model.NewBoolVar(f"x[{s}|{c}|{k}]")
     for s in S for c in C for k in range(K_by_c[c])}

# w[s,c,k,t] = x AND z
w = {(s,c,k,t): model.NewBoolVar(f"w[{s}|{c}|{k}|{t}]")
     for s in S for c in C for k in range(K_by_c[c]) for t in T}
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            for t in T:
                model.Add(w[(s,c,k,t)] <= x[(s,c,k)])
                model.Add(w[(s,c,k,t)] <= z[(c,k,t)])
                model.Add(w[(s,c,k,t)] >= x[(s,c,k)] + z[(c,k,t)] - 1)

# Cada turma escolhe no máx. 1 slot
for c in C:
    for k in range(K_by_c[c]):
        model.Add(sum(z[(c,k,t)] for t in T) <= 1)

# Proibir 2 turmas da mesma disciplina no mesmo slot
for c in C:
    for t in T:
        model.Add(sum(z[(c,k,t)] for k in range(K_by_c[c])) <= 1)

# Limite global de turmas simultâneas por slot (descongestiona)
for t in T:
    model.Add(sum(z[(c,k,t)] for c in C for k in range(K_by_c[c])) <= SLOT_MAX)

# Anti-choque para pares com alta co-demanda
for (c1,c2) in pairs_fortes:
    for t in T:
        model.Add(
            sum(z[(c1,k,t)] for k in range(K_by_c[c1])) +
            sum(z[(c2,k,t)] for k in range(K_by_c[c2])) <= 1
        )

# Capacidade por turma
for c in C:
    for k in range(K_by_c[c]):
        model.Add(sum(x[(s,c,k)] for s in S) <= sum(z[(c,k,t)] for t in T) * CAP_c[c])

# Interesse e disponibilidade
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            model.Add(x[(s,c,k)] <= D[(s,c)])
            model.Add(x[(s,c,k)] <= sum(A_S[(s,t)] * z[(c,k,t)] for t in T))

# Sem sobreposição por slot para o mesmo aluno
for s in S:
    for t in T:
        model.Add(sum(w[(s,c,k,t)] for c in C for k in range(K_by_c[c])) <= 1)

# (NOVO) Uma turma por aluno por disciplina (evita duplicação do mesmo c)
for s in S:
    for c in C:
        model.Add(sum(x[(s,c,k)] for k in range(K_by_c[c])) <= 1)

# Limite de carga por aluno (espalha vagas)
LOAD_MAX = 3   # experimente 3–5; 3 costuma aumentar cobertura
for s in S:
    model.Add(sum(x[(s,c,k)] for c in C for k in range(K_by_c[c])) <= LOAD_MAX)

# ---------- 7) Objetivo ----------
def is_noturno(tname: str) -> bool:
    txt = str(tname)
    return ("19" in txt) or ("21" in txt)

lambda_noct = 0.5   # penaliza levemente fora do noturno
obj_terms = []
# satisfação (W * x)
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            obj_terms.append(W[(s,c)] * x[(s,c,k)])
# penalidade fora do noturno
for c in C:
    for k in range(K_by_c[c]):
        for t in T:
            if not is_noturno(t):
                obj_terms.append(-lambda_noct * z[(c,k,t)])
# bônus suave por slots com alta disponibilidade
if T:
    max_disp = max(sum(A_S[(s,t)] for s in S) for t in T)
    bonus_slot = {t: (sum(A_S[(s,t)] for s in S) / max_disp) for t in T}
    slot_weight = 0.05
    for c in C:
        for k in range(K_by_c[c]):
            for t in T:
                obj_terms.append(slot_weight * bonus_slot[t] * z[(c,k,t)])

model.Maximize(sum(obj_terms))

# ---------- 8) Solver ----------
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 360.0  # 6 min
solver.parameters.num_search_workers = 8
status = solver.Solve(model)
print("\nStatus:", solver.StatusName(status))

# ---------- 9) Export ----------
OUT_DIR = Path("/content"); OUT_DIR.mkdir(parents=True, exist_ok=True)
rows_sched = []
for c in C:
    for k in range(K_by_c[c]):
        chosen = [t for t in T if solver.Value(z[(c,k,t)])==1]
        rows_sched.append({"disciplina": c, "turma": k, "slots_escolhidos": ", ".join(map(str, chosen)) if chosen else ""})
df_sched = pd.DataFrame(rows_sched)
df_sched.to_csv(OUT_DIR / "resultado_schedule.csv", index=False)

rows_mat = []
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            if solver.Value(x[(s,c,k)])==1:
                rows_mat.append({"aluno_idx": s, "disciplina": c, "turma": k})
df_mat = pd.DataFrame(rows_mat)
df_mat.to_csv(OUT_DIR / "resultado_matriculas.csv", index=False)

print("\nArquivos gerados:")
print(" -", OUT_DIR / "resultado_schedule.csv")
print(" -", OUT_DIR / "resultado_matriculas.csv")

try:
    from IPython.display import display
    print("\nPrévia do schedule:")
    display(df_sched.head(12))
    print("\nPrévia das matrículas:")
    display(df_mat.head(12))
except Exception:
    pass

# ---------- 10) KPIs ----------
total_interesses = sum(D[(s,c)] for s in S for c in C)
atendidos = 0
for s in S:
    for c in C:
        if D[(s,c)]==1 and any(solver.Value(x[(s,c,k)])==1 for k in range(K_by_c[c])):
            atendidos += 1
pct = (atendidos/total_interesses*100) if total_interesses else 0.0
turmas_abertas = sum(1 for c in C for k in range(K_by_c[c]) if any(solver.Value(z[(c,k,t)]) for t in T))

print("\n--- KPIs ---")
print("Turmas abertas:", turmas_abertas)
print("Total de matrículas:", len(df_mat))
print(f"% de interesses atendidos: {pct:.1f}%")

# ===== KPI extra: % de NOTURNO (oferta e matrículas) =====

# 1) Mapa (c,k) -> slot escolhido
slot_of = {}
for c in C:
    for k in range(K_by_c[c]):
        chosen = [t for t in T if solver.Value(z[(c,k,t)]) == 1]
        slot_of[(c,k)] = chosen[0] if chosen else None

# 2) % de turmas abertas no noturno
turmas_abertas = sum(1 for ck, t in slot_of.items() if t is not None)
turmas_noturno = sum(1 for ck, t in slot_of.items() if (t is not None and is_noturno(t)))
pct_turmas_noturno = (100 * turmas_noturno / turmas_abertas) if turmas_abertas else 0.0

# 3) % de matrículas no noturno
total_mats = 0
mats_noturno = 0
for s in S:
    for c in C:
        for k in range(K_by_c[c]):
            if solver.Value(x[(s,c,k)]) == 1:
                total_mats += 1
                tck = slot_of[(c,k)]
                if tck is not None and is_noturno(tck):
                    mats_noturno += 1
pct_mats_noturno = (100 * mats_noturno / total_mats) if total_mats else 0.0

print("\n--- KPI Noturno ---")
print(f"Turmas abertas no noturno: {turmas_noturno}/{turmas_abertas}  ({pct_turmas_noturno:.1f}%)")
print(f"Matrículas no noturno:     {mats_noturno}/{total_mats}      ({pct_mats_noturno:.1f}%)")

# (Opcional) anotar o período nos CSVs já gerados:
if 'df_sched' in globals():
    df_sched['periodo'] = df_sched['slots_escolhidos'].apply(
        lambda s: "Noturno" if any(is_noturno(t.strip()) for t in str(s).split(",") if t) else "Diurno/Outro"
    )
    df_sched.to_csv(OUT_DIR / "resultado_schedule.csv", index=False)

if 'df_mat' in globals():
    # adiciona a coluna 'slot' para cada matrícula, e o rótulo Noturno/Diurno
    def slot_for_row(row):
        return slot_of.get((row['disciplina'], row['turma']), None)
    # se df_mat tem colunas 'disciplina' e 'turma' como strings/ints, ajusta o acesso:
    df_mat['slot'] = [slot_of[(row['disciplina'], row['turma'])] for _, row in df_mat.iterrows()]
    df_mat['periodo'] = df_mat['slot'].apply(lambda t: "Noturno" if (t is not None and is_noturno(t)) else "Diurno/Outro")
    df_mat.to_csv(OUT_DIR / "resultado_matriculas.csv", index=False)



Usando arquivo: /content/respostas_formulario.xlsx
Prévia dos dados:


,id,email,turno,termo,curso_atual,curso_pretendido,horario,disciplinas_semestre_par,disciplinas_semestre_impar,disciplinas_interesse
0,3,guilherme.campos24@unifesp.br,integral,1,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos de Bioinformática"",""Algoritmos e ...","[""Algoritmos de Bioinformática"",""Algoritmos e ...","[""Nenhuma""]"
1,9,ruan.carvalho08@unifesp.br,integral,2,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Séries ...","[""Cálculo em várias variáveis"",""Probabilidade ...","[""Ciência de Dados II""]"
2,10,gcsilva06@unifesp.br,integral,4,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados II"",""Banco ...","[""Análise Real I"",""Álgebra Linear Computaciona...","[""Resolução de Problemas via Modelagem Matemát..."
3,11,jmc.nordemann@unifesp.br,noturno,6,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Programação Orientada a Objetos"",""Projeto e ...","[""Prática em Projetos Extensionistas I"",""Circu...",[]
4,12,maria.chain@unifesp.br,integral,2,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Fenômen...","[""Algoritmos e Estruturas de Dados II"",""Probab...","[""Ciência de Dados II"",""Comunicação Científica..."
5,16,nichollas.lohan@unifesp.br,integral,7,BCIT,ECOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Algoritmos e Estruturas de Dados I"",""Análise...","[""Algoritmos e Estruturas de Dados I"",""Álgebra...","[""Simulação Computacional Aplicada em Saúde"",""..."
6,24,evsribeiro@unifesp.br,noturno,2,BCIT,Só BCT,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Anatomia"",""Ciência, Tecnologia, Sociedade e ...",[],[]
7,25,felipe.bleck@unifesp.br,integral,4,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Algoritmos e Estruturas de Dados I"",""Análise...","[""Algoritmos e Estruturas de Dados I"",""Circuit...","[""Otimização e Dinâmica em Economia""]"
8,26,vitor.peneluppi@unifesp.br,integral,6,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":true,""Terça...","[""Engenharia de Software"",""Compiladores"",""Proj...",[],[]
9,35,joao.zampoli@unifesp.br,integral,6,BCIT,CCOMP,"[{""time"":""08h00 - 10h00"",""Segunda"":false,""Terç...","[""Programação Concorrente e Distribuída"",""Rede...","[""Projeto e Análise de Algoritmos"",""Cálculo em...",[]


Total de respostas lidas: 120

Slots detectados (|T|=36): ['Seg 08-10', 'Seg 10-12', 'Seg 13-15', 'Seg 15-17', 'Seg 19-21', 'Seg 21-23', 'Ter 08-10', 'Ter 10-12', 'Ter 13-15', 'Ter 15-17', 'Ter 19-21', 'Ter 21-23'] ...

Disponibilidade por slot (top 10):


,0
Seg 15-17,86
Qua 15-17,86
Qui 15-17,84
Ter 15-17,82
Qui 19-21,79
Ter 19-21,78
Qua 19-21,77
Seg 19-21,77
Sex 15-17,75
Qui 21-23,72



Total de disciplinas detectadas (|C|=198). Exemplos: ['Algoritmos de Bioinformática', 'Algoritmos e Estruturas de Dados I', 'Algoritmos e Estruturas de Dados II', 'Algoritmos em Bioinformatica', 'Anatomia', 'Análise Real I', 'Análise Real II', 'Análise de Investimento e Risco', 'Análise de Sinais', 'Aplicações de Sistemas Dinâmicos a Ciência e Tecnologia', 'Aprendizado de Máquina e Reconhecimento de Padrões', 'Arquitetura e organização de Computadores'] ...

Top 10 demandas e turmas:
- Cálculo em várias variáveis: demanda=37, cap=60, turmas=2
- Séries e Equações Diferenciais Ordinárias: demanda=37, cap=50, turmas=2
- Algoritmos e Estruturas de Dados II: demanda=31, cap=60, turmas=1
- Algoritmos e Estruturas de Dados I: demanda=29, cap=60, turmas=1
- Geometria Analítica: demanda=28, cap=50, turmas=1
- Cálculo Numérico: demanda=26, cap=60, turmas=1
- Circuitos Digitais: demanda=25, cap=50, turmas=1
- Ciência de Dados II: demanda=25, cap=50, turmas=1
- Ciência, Tecnologia, Sociedade e Am

,disciplina,turma,slots_escolhidos
0,Algoritmos de Bioinformática,0,Seg 08-10
1,Algoritmos e Estruturas de Dados I,0,Qui 19-21
2,Algoritmos e Estruturas de Dados II,0,Qua 13-15
3,Algoritmos em Bioinformatica,0,Ter 21-23
4,Anatomia,0,Sex 21-23
5,Análise Real I,0,Seg 13-15
6,Análise Real II,0,Qua 21-23
7,Análise de Investimento e Risco,0,Seg 19-21
8,Análise de Sinais,0,Qui 08-10
9,Aplicações de Sistemas Dinâmicos a Ciência e T...,0,Qui 19-21



Prévia das matrículas:


,aluno_idx,disciplina,turma
0,0,Algoritmos de Bioinformática,0
1,1,Ciência de Dados II,0
2,1,Fenômenos Mecânicos,0
3,1,Geometria Analítica,0
4,2,Algoritmos e Estruturas de Dados II,0
5,2,Cálculo Numérico,0
6,2,Otimização Linear,0
7,3,Laboratório de Sistemas Computacionais: Arquit...,0
8,3,Programação Orientada a Objetos,0
9,3,Projeto e Análise de Algoritmos,0



--- KPIs ---
Turmas abertas: 200
Total de matrículas: 326
% de interesses atendidos: 24.6%

--- KPI Noturno ---
Turmas abertas no noturno: 178/200  (89.0%)
Matrículas no noturno:     192/326      (58.9%)
